In [ ]:
# repo-root bootstrap — relative data/ paths resolve from any kernel cwd
import os, pathlib
_root = pathlib.Path.cwd()
while not (_root / 'pyproject.toml').exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)
print('cwd ->', _root)

# Human Validation + LLM-as-Judge for ACC Extraction

This notebook prepares reviewer-facing evaluation materials for Atomic Contribution Claims (ACCs):

1. Build a balanced human-annotation sample from `data/claims/openrouter_claims.csv` (positive pool) and a reusable curated bad-claims pool file (negative pool).
2. Export a blinded annotator sheet (Google Sheets-ready) and a master sheet.
3. Run an independent LLM judge on the same items.
4. Compute agreement metrics once human labels are collected.

Design goals: professional, minimal, reproducible.

## 1) Environment Setup and Dependency Imports


In [1]:
import asyncio
import hashlib
import json
import math
import os
import random
import re
import time
import unicodedata
import uuid
import warnings
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from openai import AsyncOpenAI
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

print("Imports ready.")

Imports ready.


## 2) Project Configuration and Constants

In [ ]:
@dataclass
class RunConfig:
    random_seed: int = 42
    target_total: int = 180
    year_weights: Dict[int, float] = None

    bad_claim_ratio: float = 0.40
    group_fraction: float = 0.60
    min_group_size: int = 2
    max_group_size: int = 5

    bad_roles: Tuple[str, ...] = ("Background", "Motivation", "Related Work")

    # LLM-as-judge
    judge_model: str = "google/gemini-2.5-flash-lite"
    judge_temperature: float = 0.0
    judge_top_p: float = 1.0
    judge_concurrency: int = 5
    judge_rate_per_sec: float = 5.0
    judge_checkpoint_every: int = 1

    openrouter_base_url: str = "https://openrouter.ai/api/v1"
    openrouter_api_key_env: str = "OPENROUTER_API_KEY"

    # Paths
    good_claims_path: str = "data/claims/openrouter_claims.csv"
    bad_claims_path: str = "artifacts/human_validation/bad_claims_pool.csv"
    raw_path: str = "data/raw/emnlp_papers_raw.csv"
    output_dir: str = "artifacts/human_validation"

    annotator_sheet_path: str = "artifacts/human_validation/annotator_sheet.csv"
    master_sheet_path: str = "artifacts/human_validation/master_sheet.csv"
    judge_results_path: str = "artifacts/human_validation/judge_results.csv"

    def __post_init__(self):
        if self.year_weights is None:
            self.year_weights = {
                2020: 0.30,
                2021: 0.10,
                2022: 0.10,
                2023: 0.10,
                2024: 0.10,
                2025: 0.30,
            }


CFG = RunConfig()
RNG = random.Random(CFG.random_seed)
Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print("Year weights:", CFG.year_weights)
print("Target total:", CFG.target_total)
print("Bad claims file:", CFG.bad_claims_path)

Configuration loaded.
Year weights: {2020: 0.3, 2021: 0.1, 2022: 0.1, 2023: 0.1, 2024: 0.1, 2025: 0.3}
Target total: 180
Bad claims file: artifacts/human_validation/bad_claims_pool.csv


## Quick Start (Concise)

Inputs (required):
- `data/claims/openrouter_claims.csv` with columns: `paper_id, year, original_title, atomic_claim`
- `artifacts/human_validation/bad_claims_pool.csv` with columns: `paper_id, year, original_title, atomic_claim, source, is_negative, claim_index, source_order, abstract`
- `data/raw/emnlp_papers_raw.csv` with columns: `paper_id, title, abstract`

Run order:
1. Run Sections 1-2 (imports + config).
2. Run pre-flight checks (next code cell).
3. Run preparation in Section 5 (`run_preparation`) to export annotation files.
4. Run sanity checks in Section 6.
5. Optionally run LLM-as-judge section.
6. After human labels are filled, run agreement analysis.

Outputs:
- `artifacts/human_validation/annotator_sheet.csv` (blinded sheet for annotators)
- `artifacts/human_validation/master_sheet.csv` (internal metadata)
- `artifacts/human_validation/judge_results.csv` (if judge is run)

In [3]:
def run_preflight_checks(cfg: RunConfig) -> None:
    # Required inputs ship in every bundle (public + private).
    required_paths = [cfg.good_claims_path]
    missing_required = [p for p in required_paths if not Path(p).exists()]
    if missing_required:
        raise FileNotFoundError(f"Missing required input files: {missing_required}")

    # Optional internal inputs — absent from the public release bundle (see README §Data).
    optional_paths = {
        "bad_claims_pool": cfg.bad_claims_path,
        "raw_abstracts": cfg.raw_path,
        "master_sheet": cfg.master_sheet_path,
    }
    missing_optional = {k: v for k, v in optional_paths.items() if not Path(v).exists()}
    if missing_optional:
        print(f"[skipped] optional internal inputs not shipped in the public bundle — sampling/sheet-construction and judge-vs-human sections will self-skip: {missing_optional}")

    # Bad-pool column contract (only when the pool is present).
    if Path(cfg.bad_claims_path).exists():
        bad_required = {
            "paper_id", "year", "original_title", "atomic_claim",
            "source", "is_negative", "claim_index", "source_order", "abstract",
        }
        bad_df = pd.read_csv(cfg.bad_claims_path, nrows=5)
        bad_missing = bad_required - set(bad_df.columns)
        if bad_missing:
            raise ValueError(f"bad_claims_path missing columns: {sorted(bad_missing)}")

    # Methodology provenance: ACC/judge criteria are anchored in prompts.py.
    try:
        from acc.prompts import acc_extractor_prompt, llm_as_judge_prompt

        acc_hash = hashlib.sha256(acc_extractor_prompt.encode("utf-8")).hexdigest()[:12]
        judge_hash = hashlib.sha256(llm_as_judge_prompt.encode("utf-8")).hexdigest()[:12]
        print(f"Prompt alignment -> acc_extractor_prompt hash: {acc_hash}")
        print(f"Prompt alignment -> llm_as_judge_prompt hash: {judge_hash}")
    except Exception as exc:
        print(f"Prompt alignment check skipped: {exc}")

    print("Pre-flight checks passed.")


run_preflight_checks(CFG)

# Availability flags for optional internal inputs (all present in the private repo).
HAS_RAW = Path(CFG.raw_path).exists()
HAS_BAD_POOL = Path(CFG.bad_claims_path).exists()
HAS_MASTER = Path(CFG.master_sheet_path).exists()
HAS_PREP = HAS_RAW and HAS_BAD_POOL  # sampling/sheet-construction needs both


Prompt alignment -> acc_extractor_prompt hash: f9a9b434c72c
Prompt alignment -> llm_as_judge_prompt hash: 34e9c8b0003a
Pre-flight checks passed.


## 3) Core Data Structures and Interfaces

In [4]:
@dataclass
class DataPools:
    good: pd.DataFrame
    bad: pd.DataFrame
    raw: pd.DataFrame


@dataclass
class SamplingOutputs:
    sampled: pd.DataFrame
    annotator_sheet: pd.DataFrame
    master_sheet: pd.DataFrame


class AsyncRateLimiter:
    """Simple global request pacing: at most N request starts per second."""

    def __init__(self, rate_per_sec: float):
        if rate_per_sec <= 0:
            raise ValueError("rate_per_sec must be positive")
        self._interval = 1.0 / rate_per_sec
        self._lock = asyncio.Lock()
        self._next_allowed = 0.0

    async def wait(self) -> None:
        async with self._lock:
            now = time.monotonic()
            wait_for = self._next_allowed - now
            if wait_for > 0:
                await asyncio.sleep(wait_for)
                now = time.monotonic()
            self._next_allowed = now + self._interval


print("Data structures ready.")

Data structures ready.


## 4) Utility Functions

In [5]:
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text))
    text = text.casefold()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_acc_label(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip().upper()
    if x in {"GOOD", "VALID", "YES", "1"}:
        return 1
    if x in {"BAD", "INVALID", "NO", "0"}:  # Unsure -> NA (dropped), per paper convention
        return 0
    return pd.NA

def extract_first_json_object(raw_text: str) -> Dict[str, Any]:
    raw_text = (raw_text or "").strip()
    if raw_text.startswith("```"):
        raw_text = re.sub(r"^```[a-zA-Z]*", "", raw_text).strip()
        raw_text = raw_text.rstrip("`").strip()

    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{[\s\S]*\}", raw_text)
    if not match:
        raise ValueError("No JSON object found in model output")
    return json.loads(match.group(0))


def allocate_year_targets(total: int, year_weights: Dict[int, float]) -> Dict[int, int]:
    if total <= 0:
        raise ValueError("total must be positive")
    if not year_weights:
        raise ValueError("year_weights must not be empty")

    s = sum(year_weights.values())
    if not math.isclose(s, 1.0, rel_tol=1e-6, abs_tol=1e-6):
        raise ValueError(f"year_weights must sum to 1.0, got {s}")

    raw = {y: total * w for y, w in year_weights.items()}
    floor_targets = {y: int(math.floor(v)) for y, v in raw.items()}
    remainder = total - sum(floor_targets.values())

    ranked = sorted(year_weights.keys(), key=lambda y: (raw[y] - floor_targets[y]), reverse=True)
    for y in ranked[:remainder]:
        floor_targets[y] += 1

    return floor_targets


def sample_individual_claims(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if n <= 0 or df.empty:
        return df.iloc[0:0].copy()

    per_paper = (
        df.groupby("paper_id", group_keys=False)
        .apply(lambda g: g.sample(n=1, random_state=seed))
        .reset_index(drop=True)
    )

    n_take = min(n, len(per_paper))
    return per_paper.sample(n=n_take, random_state=seed).reset_index(drop=True)


def sample_grouped_claims(df: pd.DataFrame, n_claims_target: int, cfg: RunConfig, seed: int) -> pd.DataFrame:
    if n_claims_target <= 0 or df.empty:
        return df.iloc[0:0].copy()

    counts = df.groupby("paper_id").size().rename("n_claims").reset_index()
    eligible_ids = counts[(counts["n_claims"] >= cfg.min_group_size) & (counts["n_claims"] <= cfg.max_group_size)]["paper_id"]

    eligible = df[df["paper_id"].isin(set(eligible_ids))].copy()
    if eligible.empty:
        return df.iloc[0:0].copy()

    paper_ids = list(eligible["paper_id"].drop_duplicates())
    random.Random(seed).shuffle(paper_ids)

    selected_groups = []
    total_claims = 0
    for pid in paper_ids:
        grp = eligible[eligible["paper_id"] == pid].copy()
        selected_groups.append(grp)
        total_claims += len(grp)
        if total_claims >= n_claims_target:
            break

    if not selected_groups:
        return df.iloc[0:0].copy()

    out = pd.concat(selected_groups, ignore_index=True)
    if len(out) > n_claims_target:
        out = out.iloc[:n_claims_target].copy()
    return out.reset_index(drop=True)


print("Utility functions ready.")

Utility functions ready.


In [6]:
# Override v5: use prebuilt bad claims pool file + grouped-first ordering.
def load_data_pools(cfg: RunConfig) -> DataPools:
    df_good = pd.read_csv(cfg.good_claims_path).reset_index(drop=False).rename(columns={"index": "source_order"})
    df_raw = pd.read_csv(cfg.raw_path)

    if not os.path.exists(cfg.bad_claims_path):
        raise FileNotFoundError(
            f"Missing bad claims pool file: {cfg.bad_claims_path}. "
            "Create this file first (e.g., from your claim validation results)."
        )
    df_bad = pd.read_csv(cfg.bad_claims_path)

    required_good = {"paper_id", "year", "original_title", "atomic_claim"}
    required_bad = {"paper_id", "year", "original_title", "atomic_claim", "source", "is_negative", "claim_index", "source_order", "abstract"}
    required_raw = {"paper_id", "title", "abstract"}

    miss_good = required_good - set(df_good.columns)
    miss_bad = required_bad - set(df_bad.columns)
    miss_raw = required_raw - set(df_raw.columns)

    if miss_good:
        raise ValueError(f"Missing columns in good pool: {sorted(miss_good)}")
    if miss_bad:
        raise ValueError(f"Missing columns in bad claims file: {sorted(miss_bad)}")
    if miss_raw:
        raise ValueError(f"Missing columns in raw data: {sorted(miss_raw)}")

    if "claim_id" in df_good.columns:
        df_good = df_good.drop_duplicates(subset=["claim_id"], keep="last").copy()
    else:
        df_good = df_good.drop_duplicates(subset=["paper_id", "atomic_claim"], keep="first").copy()

    raw_core = df_raw[["paper_id", "title", "abstract"]].drop_duplicates(subset=["paper_id"])

    # Standardize good pool
    df_good["source"] = "openrouter"
    df_good["is_negative"] = False
    if "rhetorical_role" not in df_good.columns:
        df_good["rhetorical_role"] = pd.NA
    if "is_valid_atomic" not in df_good.columns:
        df_good["is_valid_atomic"] = pd.NA
    if "claim_index" not in df_good.columns:
        df_good["claim_index"] = pd.NA

    if "claim_id" not in df_good.columns:
        df_good["claim_id"] = (
            "or_"
            + df_good["paper_id"].astype(str)
            + "_"
            + df_good["atomic_claim"].astype(str).map(lambda x: hashlib.sha256(normalize_text(x).encode("utf-8")).hexdigest()[:12])
        )

    common_cols = [
        "claim_id", "paper_id", "year", "original_title", "atomic_claim", "source", "is_negative",
        "rhetorical_role", "is_valid_atomic", "claim_index", "source_order"
    ]

    for col in common_cols:
        if col not in df_good.columns:
            df_good[col] = pd.NA
        if col not in df_bad.columns:
            df_bad[col] = pd.NA

    df_good = df_good[common_cols].copy().merge(raw_core, on="paper_id", how="left")

    # Bad pool already includes abstract; fill missing from raw merge for portability.
    bad_core = df_bad[common_cols + ["abstract"]].copy()
    bad_core = bad_core.merge(raw_core, on="paper_id", how="left", suffixes=("", "_raw"))
    bad_core["abstract"] = bad_core["abstract"].fillna(bad_core["abstract_raw"])
    bad_core["title"] = bad_core["title"].fillna(bad_core["original_title"])
    bad_core = bad_core.drop(columns=[c for c in ["abstract_raw"] if c in bad_core.columns])

    df_good["original_title"] = df_good["original_title"].fillna(df_good["title"])
    bad_core["original_title"] = bad_core["original_title"].fillna(bad_core["title"])

    df_good = df_good.dropna(subset=["abstract", "atomic_claim", "paper_id", "year"]).copy()
    bad_core = bad_core.dropna(subset=["abstract", "atomic_claim", "paper_id", "year"]).copy()

    df_good["year"] = df_good["year"].astype(int)
    bad_core["year"] = bad_core["year"].astype(int)
    df_good["source_order"] = pd.to_numeric(df_good["source_order"], errors="coerce").fillna(10**9).astype(int)
    bad_core["source_order"] = pd.to_numeric(bad_core["source_order"], errors="coerce").fillna(10**9).astype(int)
    df_good["claim_index"] = pd.to_numeric(df_good["claim_index"], errors="coerce")
    bad_core["claim_index"] = pd.to_numeric(bad_core["claim_index"], errors="coerce")

    return DataPools(good=df_good, bad=bad_core, raw=raw_core)


def _sample_up_to_target(df_year: pd.DataFrame, target: int, cfg: RunConfig, seed_base: int) -> pd.DataFrame:
    if target <= 0 or df_year.empty:
        return df_year.iloc[0:0].copy()

    n_group_target = int(round(target * cfg.group_fraction))
    grp = sample_grouped_claims(df_year, n_group_target, cfg, seed_base)
    used = set(grp["paper_id"]) if not grp.empty else set()

    ind = sample_individual_claims(
        df_year[~df_year["paper_id"].isin(used)],
        n=max(0, target - len(grp)),
        seed=seed_base + 1,
    )

    grp["annotation_mode"] = "grouped"
    ind["annotation_mode"] = "individual"

    out = pd.concat([grp, ind], ignore_index=True)
    if len(out) > target:
        out = out.sample(n=target, random_state=seed_base + 2).reset_index(drop=True)
    return out


def build_balanced_sample(cfg: RunConfig, pools: DataPools) -> pd.DataFrame:
    year_targets = allocate_year_targets(cfg.target_total, cfg.year_weights)
    all_year_rows = []

    for year, n_year in year_targets.items():
        desired_bad = int(round(n_year * cfg.bad_claim_ratio))
        bad_year = pools.bad[pools.bad["year"] == year].copy()
        good_year = pools.good[pools.good["year"] == year].copy()

        bad_rows = _sample_up_to_target(bad_year, desired_bad, cfg, cfg.random_seed + 1000 + year)
        bad_rows["source"] = "validation_invalid"
        bad_rows["is_negative"] = True

        n_good_needed = n_year - len(bad_rows)
        good_rows = _sample_up_to_target(good_year, n_good_needed, cfg, cfg.random_seed + 2000 + year)
        good_rows["source"] = "openrouter"
        good_rows["is_negative"] = False

        year_df = pd.concat([good_rows, bad_rows], ignore_index=True)
        if len(year_df) < n_year:
            used_claims = set(year_df["claim_id"]) if "claim_id" in year_df.columns else set()
            supplement = good_year[~good_year["claim_id"].isin(used_claims)] if "claim_id" in good_year.columns else good_year
            extra = sample_individual_claims(supplement, n_year - len(year_df), cfg.random_seed + 3000 + year)
            if not extra.empty:
                extra["annotation_mode"] = "individual"
                extra["source"] = "openrouter"
                extra["is_negative"] = False
                year_df = pd.concat([year_df, extra], ignore_index=True)

        year_df = year_df.sample(frac=1.0, random_state=cfg.random_seed + year).reset_index(drop=True)
        if len(year_df) > n_year:
            year_df = year_df.iloc[:n_year].copy()

        all_year_rows.append(year_df)

    sampled = pd.concat(all_year_rows, ignore_index=True) if all_year_rows else pd.DataFrame()
    if sampled.empty:
        raise ValueError("Sampling produced an empty DataFrame.")

    sampled = sampled.sample(frac=1.0, random_state=cfg.random_seed).reset_index(drop=True)

    def _group_id(row: pd.Series) -> Optional[str]:
        if row["annotation_mode"] != "grouped":
            return None
        digest = hashlib.sha256(str(row["paper_id"]).encode("utf-8")).hexdigest()[:8]
        return f"g_{digest}"

    sampled["group_id"] = sampled.apply(_group_id, axis=1)
    sampled["paper_title"] = sampled["original_title"].fillna(sampled["title"]).fillna("")

    return sampled

## 5) Execution Flow Wiring

In [7]:
def build_annotation_views(sampled: pd.DataFrame, cfg: RunConfig) -> SamplingOutputs:
    # Build stable per-group ordering, then shuffle all annotation units together.
    s = sampled.copy()
    s["_mode_sort"] = np.where(s["annotation_mode"] == "grouped", 0, 1)
    s["_group_sort"] = s["group_id"].fillna("zzzzzz")
    s["_in_group_order"] = pd.to_numeric(s.get("source_order", pd.Series(np.nan, index=s.index)), errors="coerce")
    s["_in_group_order"] = s["_in_group_order"].fillna(10**9)

    # For validation negatives where source_order may be from validation file, claim_index is often better in-paper order.
    if "claim_index" in s.columns:
        claim_idx = pd.to_numeric(s["claim_index"], errors="coerce")
        s["_in_group_order"] = np.where(claim_idx.notna(), claim_idx, s["_in_group_order"])

    s = s.sort_values(
        by=["_mode_sort", "_group_sort", "paper_id", "_in_group_order", "year"],
        ascending=[True, True, True, True, True],
        kind="mergesort",
    ).reset_index(drop=True)

    # Unit of shuffling:
    # - grouped claims: one full group block
    # - individual claims: one-row block
    blocks: List[pd.DataFrame] = []
    i = 0
    while i < len(s):
        row = s.iloc[i]
        if row["annotation_mode"] == "grouped" and pd.notna(row["group_id"]):
            j = i + 1
            while (
                j < len(s)
                and s.iloc[j]["annotation_mode"] == "grouped"
                and s.iloc[j]["group_id"] == row["group_id"]
            ):
                j += 1
            blocks.append(s.iloc[i:j].copy())
            i = j
        else:
            blocks.append(s.iloc[i : i + 1].copy())
            i += 1

    # Deterministic global shuffle of all units together.
    rng = random.Random(cfg.random_seed)
    rng.shuffle(blocks)

    s = pd.concat(blocks, ignore_index=True) if blocks else s.iloc[0:0].copy()

    # Reassign item_id after final ordering so sheet order and IDs match.
    s["item_id"] = np.arange(1, len(s) + 1)

    master_cols = [
        "item_id", "group_id", "annotation_mode", "source", "is_negative", "claim_id", "paper_id", "year",
        "paper_title", "abstract", "atomic_claim", "rhetorical_role", "is_valid_atomic", "claim_index", "source_order"
    ]
    existing_master_cols = [c for c in master_cols if c in s.columns]
    master = s[existing_master_cols].copy()

    annotator = master[["item_id", "group_id", "paper_title", "abstract", "atomic_claim"]].copy()
    annotator["label"] = ""
    annotator["notes"] = ""

    forbidden = {"source", "is_negative", "paper_id", "claim_id"}
    leakage = forbidden.intersection(set(annotator.columns))
    if leakage:
        raise ValueError(f"Annotator sheet leaked hidden columns: {sorted(leakage)}")

    return SamplingOutputs(sampled=s.drop(columns=["_mode_sort", "_group_sort", "_in_group_order"], errors="ignore"), annotator_sheet=annotator, master_sheet=master)


def export_sampling_outputs(outputs: SamplingOutputs, cfg: RunConfig) -> None:
    Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
    outputs.annotator_sheet.to_csv(cfg.annotator_sheet_path, index=False)
    outputs.master_sheet.to_csv(cfg.master_sheet_path, index=False)


def run_preparation(cfg: RunConfig) -> SamplingOutputs:
    pools = load_data_pools(cfg)
    sampled = build_balanced_sample(cfg, pools)
    outputs = build_annotation_views(sampled, cfg)
    export_sampling_outputs(outputs, cfg)

    print("=== PREPARATION COMPLETE ===")
    print(f"Total sampled claims: {len(outputs.sampled)}")
    print(f"Annotator sheet: {cfg.annotator_sheet_path}")
    print(f"Master sheet   : {cfg.master_sheet_path}")

    print("\nYear distribution (%):")
    year_pct = outputs.sampled["year"].value_counts(normalize=True).sort_index().mul(100).round(1)
    print(year_pct.to_string())

    print("\nSource distribution (%):")
    src_pct = outputs.sampled["source"].value_counts(normalize=True).mul(100).round(1)
    print(src_pct.to_string())

    print("\nAnnotation mode distribution (%):")
    mode_pct = outputs.sampled["annotation_mode"].value_counts(normalize=True).mul(100).round(1)
    print(mode_pct.to_string())

    return outputs


if HAS_PREP:
    prep_outputs = run_preparation(CFG)
    display(prep_outputs.sampled.head(5))
else:
    print("[skipped] data/raw abstracts and bad_claims_pool.csv not shipped in the public bundle — sampling/sheet-construction skipped; see README §Data")

=== PREPARATION COMPLETE ===
Total sampled claims: 180
Annotator sheet: artifacts/human_validation/annotator_sheet.csv
Master sheet   : artifacts/human_validation/master_sheet.csv

Year distribution (%):
year
2020    30.0
2021    10.0
2022    10.0
2023    10.0
2024    10.0
2025    30.0

Source distribution (%):
source
openrouter            75.6
validation_invalid    24.4

Annotation mode distribution (%):
annotation_mode
grouped       52.2
individual    47.8


,claim_id,paper_id,year,original_title,atomic_claim,source,is_negative,rhetorical_role,is_valid_atomic,claim_index,source_order,title,abstract,annotation_mode,group_id,paper_title,item_id
0,clm_079ab1629a6908c6,NaN,2023,Investigating Bias in Multilingual Language Mo...,Debiasing techniques with additional pretraini...,openrouter,False,<NA>,<NA>,NaN,1650,Investigating Bias in Multilingual Language Mo...,This paper investigates the transferability of...,individual,NaN,Investigating Bias in Multilingual Language Mo...,1
1,clm_39e23e2e586a6f70,NaN,2025,Reason to Rote: Rethinking Memorization in Rea...,Language models continue to compute intermedia...,openrouter,False,<NA>,<NA>,NaN,16520,Reason to Rote: Rethinking Memorization in Rea...,Large language models readily memorize arbitra...,individual,NaN,Reason to Rote: Rethinking Memorization in Rea...,2
2,clm_27b762aa7eefe926,NaN,2020,Multi-turn Response Selection using Dialogue D...,Dependency relations are helpful for dialogue ...,openrouter,False,<NA>,<NA>,NaN,4425,Multi-turn Response Selection using Dialogue D...,Multi-turn response selection is a task design...,individual,NaN,Multi-turn Response Selection using Dialogue D...,3
3,val_2020.emnlp-main.408_9021ed508c87,NaN,2020,Conversational Semantic Parsing,The proposed Seq2Seq models achieve state-of-t...,validation_invalid,True,<NA>,<NA>,5.0,168,Conversational Semantic Parsing,The structured representation for semantic par...,individual,NaN,Conversational Semantic Parsing,4
4,val_2025.emnlp-main.292_eb8acddd29d1,NaN,2025,OmniEval: An Omnidirectional and Automatic RAG...,Natural language processing remains challengin...,validation_invalid,True,<NA>,<NA>,3.0,897,OmniEval: An Omnidirectional and Automatic RAG...,Retrieval-augmented generation (RAG) has emerg...,individual,NaN,OmniEval: An Omnidirectional and Automatic RAG...,5


In [8]:
if not (HAS_PREP):
    print('[skipped] sampling pools (data/raw + bad_claims_pool.csv) not shipped in the public bundle — see README §Data')
else:
    # Optional diagnostics for pool coverage by year
    _pools = load_data_pools(CFG)
    print("Good pool by year:")
    print(_pools.good["year"].value_counts().sort_index().to_string())
    print("\nBad pool by year (from bad claims file):")
    if _pools.bad.empty:
        print("(empty)")
    else:
        print(_pools.bad["year"].value_counts().sort_index().to_string())

    print(f"\nBad pool total rows: {len(_pools.bad)}")
    print(f"Bad claims file path: {CFG.bad_claims_path}")

Good pool by year:
year
2020    2678
2021    2969
2022    3065
2023    3104
2024    3072
2025    3405

Bad pool by year (from bad claims file):
year
2020    71
2025    63

Bad pool total rows: 134
Bad claims file path: artifacts/human_validation/bad_claims_pool.csv


## 6) Sanity Checks and Assertions

In [9]:
def assert_sampling_quality(outputs: SamplingOutputs, cfg: RunConfig) -> None:
    s = outputs.sampled

    assert not s.empty, "Sample must not be empty"
    assert s["item_id"].is_unique, "item_id must be unique"
    assert s["atomic_claim"].notna().all(), "atomic_claim has NaN"
    assert s["abstract"].notna().all(), "abstract has NaN"

    target = allocate_year_targets(cfg.target_total, cfg.year_weights)
    observed = s["year"].value_counts().to_dict()

    print("Year targets vs observed:")
    for y in sorted(target.keys()):
        t = target[y]
        o = int(observed.get(y, 0))
        print(f"  {y}: target={t}, observed={o}, delta={o - t}")

    if len(s) < cfg.target_total:
        print(
            f"Warning: requested {cfg.target_total} items, produced {len(s)}. "
            "This can happen if some year/source pools are too small."
        )

    source_share = s["is_negative"].mean()
    print(f"Negative share observed: {source_share:.3f} (target {cfg.bad_claim_ratio:.3f})")
    if source_share + 1e-9 < cfg.bad_claim_ratio:
        print(
            "Warning: observed negative share is below target. "
            f"Likely cause: limited invalid rows in {cfg.bad_claims_path} under year constraints."
        )

    leaked = {"source", "is_negative", "paper_id", "claim_id"}.intersection(set(outputs.annotator_sheet.columns))
    assert not leaked, f"Annotator sheet leakage detected: {sorted(leaked)}"


if HAS_PREP:
    assert_sampling_quality(prep_outputs, CFG)
    print("Sanity checks passed.")
else:
    print("[skipped] sampling not run in the public bundle (no raw/bad pool) — sanity checks skipped; see README §Data")

Year targets vs observed:
  2020: target=54, observed=54, delta=0
  2021: target=18, observed=18, delta=0
  2022: target=18, observed=18, delta=0
  2023: target=18, observed=18, delta=0
  2024: target=18, observed=18, delta=0
  2025: target=54, observed=54, delta=0
Negative share observed: 0.244 (target 0.400)
Sanity checks passed.


## 7) Lightweight Unit Tests in Notebook

In [10]:
# Test 1: exact year target allocation
_t = allocate_year_targets(180, {2020: 0.30, 2021: 0.10, 2022: 0.10, 2023: 0.10, 2024: 0.10, 2025: 0.30})
assert _t == {2020: 54, 2021: 18, 2022: 18, 2023: 18, 2024: 18, 2025: 54}

# Test 2: failure case for invalid weights
try:
    _ = allocate_year_targets(100, {2020: 0.2, 2021: 0.2})
    raise AssertionError("Expected ValueError for invalid weights sum")
except ValueError:
    pass

# Test 3: annotator sheet should not leak hidden fields
if HAS_PREP:
    _leak = {"source", "is_negative", "paper_id", "claim_id"}.intersection(set(prep_outputs.annotator_sheet.columns))
    assert not _leak
else:
    print("[skipped] Test 3 needs the sampled sheet (no raw/bad pool in the public bundle); see README §Data")

print("Unit tests passed.")

Unit tests passed.


## 8) Run End-to-End Example

In [11]:
if not (HAS_PREP):
    print('[skipped] annotation sheets not generated in the public bundle (no raw/bad pool) — see README §Data')
else:
    print("Generated files:")
    print(f"- {CFG.annotator_sheet_path}")
    print(f"- {CFG.master_sheet_path}")

    display(prep_outputs.annotator_sheet.head(3))
    print("\nGrouped example rows:")
    display(prep_outputs.annotator_sheet[prep_outputs.annotator_sheet["group_id"].notna()].head(5))

Generated files:
- artifacts/human_validation/annotator_sheet.csv
- artifacts/human_validation/master_sheet.csv


,item_id,group_id,paper_title,abstract,atomic_claim,label,notes
0,1,NaN,Investigating Bias in Multilingual Language Mo...,This paper investigates the transferability of...,Debiasing techniques with additional pretraini...,,
1,2,NaN,Reason to Rote: Rethinking Memorization in Rea...,Large language models readily memorize arbitra...,Language models continue to compute intermedia...,,
2,3,NaN,Multi-turn Response Selection using Dialogue D...,Multi-turn response selection is a task design...,Dependency relations are helpful for dialogue ...,,



Grouped example rows:


,item_id,group_id,paper_title,abstract,atomic_claim,label,notes
5,6,g_9ba73056,Predicting Cross-lingual Trends in Microblogs,Trends on microblogs often transcend linguisti...,The proposed approach enables the forecasting ...,,
6,7,g_9ba73056,Predicting Cross-lingual Trends in Microblogs,Trends on microblogs often transcend linguisti...,"Many previous studies discuss this issue, and ...",,
11,12,g_fd17dcf7,Exploring the Linear Subspace Hypothesis in Ge...,Bolukbasi et al. (2016) presents one of the fi...,A kernelized non-linear technique isolates gen...,,
12,13,g_fd17dcf7,Exploring the Linear Subspace Hypothesis in Ge...,Bolukbasi et al. (2016) presents one of the fi...,The linearity of the gender bias subspace in w...,,
13,14,g_fd17dcf7,Exploring the Linear Subspace Hypothesis in Ge...,Bolukbasi et al. (2016) presents one of the fi...,Gender bias in word embeddings is well capture...,,


### LLM-as-Judge (independent model family)

Run this section after preparing `master_sheet.csv`. It can be skipped if no API key is configured.

Methodology alignment note: ACC validity criteria are anchored to `acc_extractor_prompt` in `prompts.py`, and the judge rubric in this notebook is aligned with `llm_as_judge_prompt` (same validity-vs-invalidity framing for contribution-bearing, atomicity, faithfulness, decontextualization, and exclusion of meta-language/raw-metric-only claims).

In [18]:
from acc.prompts import llm_as_judge_prompt

JUDGE_RESULT_COLUMNS = [
    "item_id",
    "judge_verdict",
    "judge_main_issue",
    "judge_reason",
    "judge_confidence",
    "judge_model",
    "judged_at",
]

JUDGE_SYSTEM_PROMPT = "You are an expert evaluator of scientific claims extracted from NLP paper abstracts."
JUDGE_TIMEOUT_SEC = 60
JUDGE_MAX_TOKENS = 400


def load_judge_results(path: str) -> pd.DataFrame:
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return pd.DataFrame(columns=JUDGE_RESULT_COLUMNS)

    df = pd.read_csv(path)
    for col in JUDGE_RESULT_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    return df[JUDGE_RESULT_COLUMNS].copy()


def atomic_write_csv(path: str, df: pd.DataFrame) -> None:
    path_obj = Path(path)
    path_obj.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = path_obj.with_name(path_obj.name + ".tmp")
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, path_obj)


def normalize_judge_response(parsed: Dict[str, Any]) -> Dict[str, str]:
    if not isinstance(parsed, dict):
        raise ValueError(f"Judge response is not a JSON object: {type(parsed)}")

    item = parsed["evaluations"][0] if isinstance(parsed.get("evaluations"), list) else parsed
    if not isinstance(item, dict):
        raise ValueError(f"Judge evaluation is not an object: {type(item)}")

    label = str(item.get("label", "")).strip().upper()
    label = {
        "VALID": "GOOD",
        "INVALID": "BAD",
        "TRUE": "GOOD",
        "FALSE": "BAD",
        "YES": "GOOD",
        "NO": "BAD",
    }.get(label, label)

    if label not in {"GOOD", "BAD", "UNSURE"}:
        raise ValueError(f"Invalid judge label: {item.get('label')!r}")

    main_issue = str(item.get("main_issue", "other")).strip().lower()
    allowed_issues = {
        "none",
        "background_context",
        "unsupported",
        "too_vague",
        "not_self_contained",
        "not_contribution",
        "overgeneralized",
        "mixed_claims",
        "other",
    }
    if main_issue not in allowed_issues:
        main_issue = "other"

    reason = str(item.get("brief_reason") or item.get("reason") or "").strip()
    if not reason:
        raise ValueError("Judge response is missing brief_reason")

    confidence = str(item.get("confidence", "medium")).strip().lower()
    if confidence not in {"high", "medium", "low"}:
        confidence = "medium"

    return {
        "judge_verdict": label,
        "judge_main_issue": main_issue,
        "judge_reason": reason,
        "judge_confidence": confidence,
    }


@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=1, min=1, max=12),
    retry=retry_if_exception_type(Exception),
)
async def judge_one_async(
    client: AsyncOpenAI,
    semaphore: asyncio.Semaphore,
    rate_limiter: AsyncRateLimiter,
    row: pd.Series,
    cfg: RunConfig,
) -> Dict[str, str]:
    user_prompt = (
        llm_as_judge_prompt
        .replace("<<TITLE>>", "" if pd.isna(row["paper_title"]) else str(row["paper_title"]))
        .replace("<<ABSTRACT>>", "" if pd.isna(row["abstract"]) else str(row["abstract"]))
        .replace("<<CLAIM>>", "" if pd.isna(row["atomic_claim"]) else str(row["atomic_claim"]))
    )

    await rate_limiter.wait()

    async with semaphore:
        response = await asyncio.wait_for(
            client.chat.completions.create(
                model=cfg.judge_model,
                temperature=cfg.judge_temperature,
                top_p=cfg.judge_top_p,
                max_tokens=JUDGE_MAX_TOKENS,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
            ),
            timeout=JUDGE_TIMEOUT_SEC,
        )

    content = response.choices[0].message.content
    if content is None:
        raise ValueError("Model returned message.content=None")

    parsed = extract_first_json_object(content)
    return normalize_judge_response(parsed)


async def run_llm_judge(cfg: RunConfig) -> pd.DataFrame:
    judge_path = Path(cfg.judge_results_path)

    if judge_path.exists():
        judge_df = load_judge_results(cfg.judge_results_path)
        print(f"LLM judge skipped: existing file found at {cfg.judge_results_path}")
        print(f"Loaded rows: {len(judge_df)}")
        return judge_df

    api_key = os.environ.get(cfg.openrouter_api_key_env)
    if not api_key:
        raise ValueError(f"Environment variable {cfg.openrouter_api_key_env} is not set")

    master = pd.read_csv(cfg.master_sheet_path)
    required_cols = {"item_id", "paper_title", "abstract", "atomic_claim"}
    missing = required_cols - set(master.columns)
    if missing:
        raise ValueError(f"Master sheet is missing columns: {sorted(missing)}")

    master = master.dropna(subset=["item_id", "atomic_claim"]).copy()
    master["item_id"] = master["item_id"].astype(int)
    master = master.drop_duplicates("item_id", keep="last").sort_values("item_id")

    client = AsyncOpenAI(
        api_key=api_key,
        base_url=cfg.openrouter_base_url,
        timeout=JUDGE_TIMEOUT_SEC,
        default_headers={
            "HTTP-Referer": "https://local-notebook.acc",
            "X-Title": "ACC Human Validation Judge",
        },
    )

    semaphore = asyncio.Semaphore(max(1, int(cfg.judge_concurrency)))
    rate_limiter = AsyncRateLimiter(float(cfg.judge_rate_per_sec))
    judged_at = datetime.now(timezone.utc).isoformat()

    async def run_row(row: pd.Series) -> Dict[str, Any]:
        judged = await judge_one_async(client, semaphore, rate_limiter, row, cfg)
        return {
            "item_id": int(row["item_id"]),
            **judged,
            "judge_model": cfg.judge_model,
            "judged_at": judged_at,
        }

    tasks = [asyncio.create_task(run_row(row)) for _, row in master.iterrows()]
    rows = []

    for task in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="LLM-as-judge"):
        rows.append(await task)

    judge_df = pd.DataFrame(rows, columns=JUDGE_RESULT_COLUMNS)
    judge_df = judge_df.sort_values("item_id").reset_index(drop=True)

    atomic_write_csv(cfg.judge_results_path, judge_df)

    print(f"LLM judge complete: saved {len(judge_df)} rows to {cfg.judge_results_path}")
    return judge_df


judge_df = await run_llm_judge(CFG)
display(judge_df.head(3))

LLM-as-judge: 100%|██████████| 180/180 [00:37<00:00,  4.83it/s]


LLM judge complete: saved 180 rows to artifacts/human_validation/judge_results.csv


,item_id,judge_verdict,judge_main_issue,judge_reason,judge_confidence,judge_model,judged_at
0,1,GOOD,none,The claim is supported by the abstract and des...,high,google/gemini-2.5-flash-lite,2026-05-26T09:26:20.469159+00:00
1,2,GOOD,none,The claim is a direct finding supported by the...,high,google/gemini-2.5-flash-lite,2026-05-26T09:26:20.469159+00:00
2,3,GOOD,none,The claim is supported by the abstract as an e...,high,google/gemini-2.5-flash-lite,2026-05-26T09:26:20.469159+00:00


### Agreement Analysis (after human annotation)

Expected: `annotator_sheet.csv` with `label` column filled as GOOD/BAD/UNSURE.

In [19]:
def compute_agreement_metrics(cfg: RunConfig, human_sheet_path: Optional[str] = None) -> pd.DataFrame:
    human_path = human_sheet_path or cfg.annotator_sheet_path
    human = pd.read_csv(human_path)
    master = pd.read_csv(cfg.master_sheet_path)
    judge = pd.read_csv(cfg.judge_results_path)

    required_human = {"item_id", "label"}
    missing_h = required_human - set(human.columns)
    if missing_h:
        raise ValueError(f"Human sheet missing columns: {sorted(missing_h)}")

    merged = (
        human.merge(master, on="item_id", how="inner")
        .merge(judge[["item_id", "judge_verdict"]], on="item_id", how="inner")
    )

    merged = merged[merged["label"].notna()].copy()
    if merged.empty:
        raise ValueError("No human labels found. Fill `label` column first.")

    merged["human_label"] = merged["label"].apply(normalize_acc_label)
    merged = merged[merged["human_label"].isin([0, 1])].copy()
    if merged.empty:
        raise ValueError("Human labels must be binary 0/1.")

    merged["judge_label"] = merged["judge_verdict"].map({"invalid": 0, "valid": 1}).astype(int)

    y_true = merged["human_label"].astype(int).to_numpy()
    y_pred = merged["judge_label"].astype(int).to_numpy()

    kappa = cohen_kappa_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)

    print("=== Human vs LLM Judge ===")
    print(f"N: {len(merged)}")
    print(f"Cohen's kappa: {kappa:.3f}")
    print(f"Accuracy: {acc:.3f}")
    print(f"Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}")
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, digits=3, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title("Human vs LLM Judge")
    ax.set_xlabel("Judge label")
    ax.set_ylabel("Human label")
    ax.set_xticks([0, 1], ["invalid", "valid"])
    ax.set_yticks([0, 1], ["invalid", "valid"])

    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", color="black")

    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    cm_path = Path(cfg.output_dir) / "confusion_matrix_human_vs_judge.png"
    fig.savefig(cm_path, dpi=150, bbox_inches="tight")
    print(f"Confusion matrix saved: {cm_path}")

    by_source = (
        merged.groupby("source")
        .apply(lambda g: pd.Series({
            "n": len(g),
            "accuracy": accuracy_score(g["human_label"], g["judge_label"]),
            "kappa": cohen_kappa_score(g["human_label"], g["judge_label"]) if g["human_label"].nunique() > 1 else np.nan,
        }))
        .reset_index()
    )

    print("\nBy source:")
    display(by_source)
    return merged


# Conditional execution: run agreement analysis only if judge results exist and human labels are available
if not HAS_MASTER:
    print("[skipped] artifacts/human_validation/master_sheet.csv not shipped in the public bundle — human-vs-judge needs master+annotator sheets; the 3-annotator agreement below is complete. See README §Data")
elif Path(CFG.judge_results_path).exists():
    print(f"→ Running agreement analysis...")
    try:
        merged_eval = compute_agreement_metrics(CFG)
        print(f"✓ Agreement analysis complete. Evaluated {len(merged_eval)} items.")
        display(merged_eval.head(3))
    except ValueError as e:
        print(f"⚠ Agreement analysis skipped: {e}")
else:
    print(f"⚠ Judge results not found at {CFG.judge_results_path}")
    print(f"  → Run the judge section first (above) or ensure the file exists.")


→ Running agreement analysis...
⚠ Agreement analysis skipped: No human labels found. Fill `label` column first.


In [ ]:
# === 3-annotator agreement + LLM-judge validation (paper Appendix numbers) ===
# Unsure/blank labels are dropped (normalize_acc_label -> NA), matching the paper
# convention. Reproduces the numbers in Section "Human Validation Protocol".
from itertools import combinations

ANNOTATOR_SHEETS = [
    "artifacts/human_validation/annotator_sheet-1.csv",
    "artifacts/human_validation/annotator_sheet-2.csv",
    "artifacts/human_validation/annotator_sheet-3.csv",
]


def _fleiss_kappa(binary_matrix) -> float:
    """Fleiss' kappa for an (N items x R raters) binary {0,1} matrix (no missing)."""
    m = np.asarray(binary_matrix, dtype=int)
    n_items, n_raters = m.shape
    counts = np.stack([(m == 0).sum(axis=1), (m == 1).sum(axis=1)], axis=1)
    P_i = ((counts ** 2).sum(axis=1) - n_raters) / (n_raters * (n_raters - 1))
    p_j = counts.sum(axis=0) / (n_items * n_raters)
    P_e = (p_j ** 2).sum()
    return (P_i.mean() - P_e) / (1 - P_e)


def human_agreement_report(sheet_paths=ANNOTATOR_SHEETS):
    """Pairwise Cohen's kappa + Fleiss' kappa across all annotator sheets."""
    frames, cols = [], []
    for i, p in enumerate(sheet_paths, 1):
        s = pd.read_csv(p)[["item_id", "label"]].set_index("item_id")["label"]
        frames.append(s.apply(normalize_acc_label).rename(f"a{i}"))
        cols.append(f"a{i}")
    M = pd.concat(frames, axis=1)

    print(f"=== Inter-annotator agreement ({len(cols)} annotators, Unsure excluded) ===")
    for a, b in combinations(cols, 2):
        d = M[[a, b]].dropna()
        print(f"  Cohen {a} vs {b}: kappa={cohen_kappa_score(d[a].astype(int), d[b].astype(int)):.3f}"
              f"  agree={(d[a] == d[b]).mean():.3f}  N={len(d)}")
    allsure = M.dropna()
    if len(cols) >= 3:
        print(f"  Fleiss kappa: {_fleiss_kappa(allsure.values):.3f}  N={len(allsure)}"
              f"  unanimous={(allsure.nunique(axis=1) == 1).mean():.3f}")
    return M


def judge_validation_report(M):
    """LLM judge vs each annotator, vs 2-of-3 majority consensus, and corpus quality."""
    judge = pd.read_csv(CFG.judge_results_path).set_index("item_id")["judge_verdict"].apply(normalize_acc_label)
    master = pd.read_csv(CFG.master_sheet_path).set_index("item_id")
    cols = list(M.columns)

    print("\n=== LLM judge vs humans (Unsure excluded) ===")
    for c in cols:
        d = pd.concat([M[c], judge.rename("j")], axis=1).dropna()
        print(f"  judge vs {c}: acc={accuracy_score(d[c].astype(int), d['j'].astype(int)):.3f}"
              f"  kappa={cohen_kappa_score(d[c].astype(int), d['j'].astype(int)):.3f}  N={len(d)}")
    allsure = M.dropna()
    majority = (allsure.sum(axis=1) >= allsure.shape[1] / 2).astype(int)
    d = pd.concat([majority.rename("g"), judge.rename("j")], axis=1).dropna()
    print(f"  judge vs MAJORITY (>= {int(np.ceil(len(cols) / 2))}/{len(cols)} Good):"
          f" acc={accuracy_score(d['g'].astype(int), d['j'].astype(int)):.3f}"
          f"  kappa={cohen_kappa_score(d['g'].astype(int), d['j'].astype(int)):.3f}  N={len(d)}")

    neg = master["is_negative"].astype(str).str.upper().isin(["TRUE", "1"])
    sampled = master.index[~neg]
    print("\n=== Sampled extracted claims (corpus quality, n={}) ===".format(len(sampled)))
    for c in cols:
        s = M[c].reindex(sampled).dropna()
        print(f"  {c}: Good {int((s == 1).sum())}/{len(s)} ({(s == 1).mean():.3f})")
    js = judge.reindex(sampled)
    print(f"  judge: Good {int((js == 1).sum())}/{js.notna().sum()} ({(js == 1).mean():.3f})")


if all(Path(p).exists() for p in ANNOTATOR_SHEETS):
    _M = human_agreement_report()
    if Path(CFG.judge_results_path).exists() and HAS_MASTER:
        judge_validation_report(_M)
    elif not HAS_MASTER:
        print("[skipped] master_sheet.csv not shipped in the public bundle — judge-vs-human/majority report needs it; inter-annotator agreement above is complete. See README §Data")
else:
    print("Annotator sheets not all present; skipping agreement report.")
